# 00 — Data Splitter Final (2 Skenario)

## Prinsip Utama
```
Test set = DATA MANUAL MURNI untuk SEMUA eksperimen
  → Apple-to-apple antar semua model
  → Bebas circular dependency
  → Ground truth = label manusia (Seprianto)
  → Referensi: Li et al. (2022); Li et al. (2024)
```

## 2 Skenario × 4 Model = 8 Eksperimen

| Kode | Model | Skenario | Train | Test |
|---|---|---|---|---|
| E-01 | LR | Manual | 80% manual | 20% manual |
| E-02 | LR | Gabungan | auto + 80% manual | 20% manual ← SAMA |
| E-03 | XGB NoTune | Manual | 80% manual | 20% manual |
| E-04 | XGB NoTune | Gabungan | auto + 80% manual | 20% manual ← SAMA |
| E-05 | XGB Tuned | Manual | 80% manual | 20% manual |
| E-06 | XGB Tuned | Gabungan | auto + 80% manual | 20% manual ← SAMA |
| E-07 | IndoBERT | Manual | 80% manual | 20% manual |
| E-08 | IndoBERT | Gabungan | auto + 80% manual | 20% manual ← SAMA |

## Output — 8 File CSV di folder `splits/`
```
splits/
  XGBoost / LR:
  ├── manual_xgb_train.csv    (11.992 baris)
  ├── manual_xgb_test.csv     ( 2.999 baris) ← dipakai E-01,03,05
  ├── gabungan_xgb_train.csv  (53.316 baris)
  └── gabungan_xgb_test.csv   ( 2.999 baris) ← dipakai E-02,04,06 (SAMA)

  IndoBERT:
  ├── manual_idb_train.csv    (12.381 baris)
  ├── manual_idb_test.csv     ( 3.096 baris) ← dipakai E-07
  ├── gabungan_idb_train.csv  (54.045 baris)
  └── gabungan_idb_test.csv   ( 3.096 baris) ← dipakai E-08 (SAMA)
```


## Cell 1 — Setup

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split

BASE_DIR  = r'C:\Users\Lenovo\Downloads\skrips_code\xgboost_indobert_method'
PREP_DIR  = os.path.join(BASE_DIR, '04_data_preprocessing')
SPLIT_DIR = os.path.join(BASE_DIR, 'splits')
os.makedirs(SPLIT_DIR, exist_ok=True)

LABEL_MAP = {'keluhan':0, 'saran':1, 'pujian':2}
SEED      = 42

print('Setup selesai!')
print(f'PREP_DIR  : {PREP_DIR}')
print(f'SPLIT_DIR : {SPLIT_DIR}')

Setup selesai!
PREP_DIR  : C:\Users\Lenovo\Downloads\skrips_code\xgboost_indobert_method\04_data_preprocessing
SPLIT_DIR : C:\Users\Lenovo\Downloads\skrips_code\xgboost_indobert_method\splits


## Cell 2 — Load 4 File Preprocessing

In [2]:
prep_manual_xgb = pd.read_csv(os.path.join(PREP_DIR, 'prep_manual_xgb.csv'))
prep_auto_xgb   = pd.read_csv(os.path.join(PREP_DIR, 'prep_auto_xgb.csv'))
prep_manual_idb = pd.read_csv(os.path.join(PREP_DIR, 'prep_manual_idb.csv'))
prep_auto_idb   = pd.read_csv(os.path.join(PREP_DIR, 'prep_auto_idb.csv'))

# Tambahkan label_enc
for df in [prep_manual_xgb, prep_auto_xgb,
           prep_manual_idb, prep_auto_idb]:
    df['label_enc'] = df['label_pks'].map(LABEL_MAP)

print('='*60)
print('DATA YANG DIMUAT')
print('='*60)
print(f'prep_manual_xgb : {len(prep_manual_xgb):,} baris ')
print(f'prep_auto_xgb   : {len(prep_auto_xgb):,} baris ')
print(f'prep_manual_idb : {len(prep_manual_idb):,} baris ')
print(f'prep_auto_idb   : {len(prep_auto_idb):,} baris ')

# Validasi semua confidence sudah benar
print(f'\nValidasi confidence:')
print(f'  manual_xgb semua conf=1.0 : {(prep_manual_xgb.confidence==1.0).all()}')
print(f'  auto_xgb   semua conf<1.0 : {(prep_auto_xgb.confidence<1.0).all()}')
print(f'  manual_idb semua conf=1.0 : {(prep_manual_idb.confidence==1.0).all()}')
print(f'  auto_idb   semua conf<1.0 : {(prep_auto_idb.confidence<1.0).all()}')

# Distribusi label
print(f'\nDistribusi label manual XGB:')
for lbl, cnt in prep_manual_xgb['label_pks'].value_counts().items():
    print(f'  {lbl:10s}: {cnt:,} ({cnt/len(prep_manual_xgb)*100:.1f}%)')

print(f'\nDistribusi label auto XGB (band tinggi):')
for lbl, cnt in prep_auto_xgb['label_pks'].value_counts().items():
    print(f'  {lbl:10s}: {cnt:,} ({cnt/len(prep_auto_xgb)*100:.1f}%)')

DATA YANG DIMUAT
prep_manual_xgb : 14,991 baris 
prep_auto_xgb   : 41,324 baris 
prep_manual_idb : 15,477 baris 
prep_auto_idb   : 41,664 baris 

Validasi confidence:
  manual_xgb semua conf=1.0 : True
  auto_xgb   semua conf<1.0 : True
  manual_idb semua conf=1.0 : True
  auto_idb   semua conf<1.0 : True

Distribusi label manual XGB:
  keluhan   : 9,834 (65.6%)
  saran     : 2,671 (17.8%)
  pujian    : 2,486 (16.6%)

Distribusi label auto XGB (band tinggi):
  keluhan   : 29,085 (70.4%)
  pujian    : 7,014 (17.0%)
  saran     : 5,225 (12.6%)


## Cell 3 — Split Data Manual 80:20
**Test set ini digunakan untuk SEMUA 8 eksperimen.**
**Alasan:** evaluasi terhadap ground truth label manusia,
bebas dari kontaminasi pseudo-label (Li et al., 2022; Li et al., 2024)


In [3]:
# ── Split manual XGBoost ─────────────────────────────────────────
man_xgb_train, man_xgb_test = train_test_split(
    prep_manual_xgb,
    test_size    = 0.2,
    random_state = SEED,
    stratify     = prep_manual_xgb['label_enc']
)

# ── Split manual IndoBERT ─────────────────────────────────────────
man_idb_train, man_idb_test = train_test_split(
    prep_manual_idb,
    test_size    = 0.2,
    random_state = SEED,
    stratify     = prep_manual_idb['label_enc']
)

print('='*60)
print('SPLIT DATA MANUAL (TEST SET SEMUA EKSPERIMEN)')
print('='*60)
print(f'\nXGBoost / LR:')
print(f'  Train : {len(man_xgb_train):,} baris (80%)')
print(f'  Test  : {len(man_xgb_test):,} baris (20%) ← E-01,02,03,04,05,06')

print(f'\nIndoBERT:')
print(f'  Train : {len(man_idb_train):,} baris (80%)')
print(f'  Test  : {len(man_idb_test):,} baris (20%) ← E-07,08')

print(f'\nDistribusi test set XGB (label manusia):')
for lbl, cnt in man_xgb_test['label_pks'].value_counts().items():
    pct = cnt/len(man_xgb_test)*100
    bar = chr(9608)*int(pct/3)
    print(f'  {lbl:10s}: {cnt:5,} ({pct:.1f}%) {bar}')

print(f'\nDistribusi test set IDB (label manusia):')
for lbl, cnt in man_idb_test['label_pks'].value_counts().items():
    pct = cnt/len(man_idb_test)*100
    bar = chr(9608)*int(pct/3)
    print(f'  {lbl:10s}: {cnt:5,} ({pct:.1f}%) {bar}')

SPLIT DATA MANUAL (TEST SET SEMUA EKSPERIMEN)

XGBoost / LR:
  Train : 11,992 baris (80%)
  Test  : 2,999 baris (20%) ← E-01,02,03,04,05,06

IndoBERT:
  Train : 12,381 baris (80%)
  Test  : 3,096 baris (20%) ← E-07,08

Distribusi test set XGB (label manusia):
  keluhan   : 1,967 (65.6%) █████████████████████
  saran     :   535 (17.8%) █████
  pujian    :   497 (16.6%) █████

Distribusi test set IDB (label manusia):
  keluhan   : 2,051 (66.2%) ██████████████████████
  saran     :   552 (17.8%) █████
  pujian    :   493 (15.9%) █████


## Cell 4 — Skenario Gabungan
```
Train = data otomatis (band tinggi) + 80% manual
Test  = 20% manual (SAMA dengan skenario manual)
```


In [4]:
# ── Gabungan XGBoost ─────────────────────────────────────────────
# Train = semua auto + 80% manual
gab_xgb_train = pd.concat(
    [prep_auto_xgb, man_xgb_train],
    ignore_index=True
).sample(frac=1, random_state=SEED).reset_index(drop=True)  # acak urutan

# Test = 20% manual (SAMA dengan skenario manual)
gab_xgb_test  = man_xgb_test.copy()

# ── Gabungan IndoBERT ─────────────────────────────────────────────
gab_idb_train = pd.concat(
    [prep_auto_idb, man_idb_train],
    ignore_index=True
).sample(frac=1, random_state=SEED).reset_index(drop=True)  # acak urutan

gab_idb_test  = man_idb_test.copy()

print('='*60)
print('SKENARIO GABUNGAN')
print('='*60)
print(f'\nXGBoost / LR — Gabungan:')
print(f'  Train auto   : {len(prep_auto_xgb):,}')
print(f'  Train manual : {len(man_xgb_train):,}')
print(f'  Train TOTAL  : {len(gab_xgb_train):,}')
print(f'  Test         : {len(gab_xgb_test):,} ← manual murni')

print(f'\nIndoBERT — Gabungan:')
print(f'  Train auto   : {len(prep_auto_idb):,}')
print(f'  Train manual : {len(man_idb_train):,}')
print(f'  Train TOTAL  : {len(gab_idb_train):,}')
print(f'  Test         : {len(gab_idb_test):,} ← manual murni')

print(f'\nDistribusi label train gabungan XGB:')
for lbl, cnt in gab_xgb_train['label_pks'].value_counts().items():
    pct = cnt/len(gab_xgb_train)*100
    bar = chr(9608)*int(pct/3)
    print(f'  {lbl:10s}: {cnt:6,} ({pct:.1f}%) {bar}')

SKENARIO GABUNGAN

XGBoost / LR — Gabungan:
  Train auto   : 41,324
  Train manual : 11,992
  Train TOTAL  : 53,316
  Test         : 2,999 ← manual murni

IndoBERT — Gabungan:
  Train auto   : 41,664
  Train manual : 12,381
  Train TOTAL  : 54,045
  Test         : 3,096 ← manual murni

Distribusi label train gabungan XGB:
  keluhan   : 36,952 (69.3%) ███████████████████████
  pujian    :  9,003 (16.9%) █████
  saran     :  7,361 (13.8%) ████


## Cell 5 — Ringkasan 2 Skenario

In [5]:
print('='*70)
print('RINGKASAN FINAL 2 SKENARIO × 4 MODEL = 8 EKSPERIMEN')
print('='*70)
print(f'\n{"Kode":5s} {"Model":12s} {"Skenario":12s} '
      f'{"Train":>8s} {"Test":>6s} {"Test Sumber":>15s}')
print('-'*70)

eksperimen = [
    ('E-01','LR',         'Manual',  man_xgb_train, man_xgb_test),
    ('E-02','LR',         'Gabungan',gab_xgb_train, gab_xgb_test),
    ('E-03','XGB NoTune', 'Manual',  man_xgb_train, man_xgb_test),
    ('E-04','XGB NoTune', 'Gabungan',gab_xgb_train, gab_xgb_test),
    ('E-05','XGB Tuned',  'Manual',  man_xgb_train, man_xgb_test),
    ('E-06','XGB Tuned',  'Gabungan',gab_xgb_train, gab_xgb_test),
    ('E-07','IndoBERT',   'Manual',  man_idb_train, man_idb_test),
    ('E-08','IndoBERT',   'Gabungan',gab_idb_train, gab_idb_test),
]
for kode, model, ske, tr, te in eksperimen:
    src = 'Manual murni ✅'
    print(f'{kode:5s} {model:12s} {ske:12s} '
          f'{len(tr):8,} {len(te):6,} {src:>15s}')

print(f'\n{"─"*70}')
print('Konfirmasi:')
print(f'  XGB/LR Manual test   = XGB/LR Gabungan test : '
      f'{set(man_xgb_test.id.astype(str)) == set(gab_xgb_test.id.astype(str))}')
print(f'  IDB Manual test      = IDB Gabungan test    : '
      f'{set(man_idb_test.id.astype(str)) == set(gab_idb_test.id.astype(str))}')
print(f'\n→ Semua test set identik per model = benar-benar apple-to-apple!')
print(f'→ Semua test set = label manusia   = bebas circular dependency!')

RINGKASAN FINAL 2 SKENARIO × 4 MODEL = 8 EKSPERIMEN

Kode  Model        Skenario        Train   Test     Test Sumber
----------------------------------------------------------------------
E-01  LR           Manual         11,992  2,999  Manual murni ✅
E-02  LR           Gabungan       53,316  2,999  Manual murni ✅
E-03  XGB NoTune   Manual         11,992  2,999  Manual murni ✅
E-04  XGB NoTune   Gabungan       53,316  2,999  Manual murni ✅
E-05  XGB Tuned    Manual         11,992  2,999  Manual murni ✅
E-06  XGB Tuned    Gabungan       53,316  2,999  Manual murni ✅
E-07  IndoBERT     Manual         12,381  3,096  Manual murni ✅
E-08  IndoBERT     Gabungan       54,045  3,096  Manual murni ✅

──────────────────────────────────────────────────────────────────────
Konfirmasi:
  XGB/LR Manual test   = XGB/LR Gabungan test : True
  IDB Manual test      = IDB Gabungan test    : True

→ Semua test set identik per model = benar-benar apple-to-apple!
→ Semua test set = label manusia   = bebas c

## Cell 6 — Simpan 8 File CSV

In [ ]:
splits = {
    # ── Skenario Manual ───────────────────────────────────────────
    'manual_xgb_train'   : man_xgb_train,
    'manual_xgb_test'    : man_xgb_test,
    'manual_idb_train'   : man_idb_train,
    'manual_idb_test'    : man_idb_test,

    # ── Skenario Gabungan ─────────────────────────────────────────
    'gabungan_xgb_train' : gab_xgb_train,
    'gabungan_xgb_test'  : gab_xgb_test,
    'gabungan_idb_train' : gab_idb_train,
    'gabungan_idb_test'  : gab_idb_test,
}

print('Menyimpan 8 file CSV...')
print('='*65)
for nama, df in splits.items():
    path = os.path.join(SPLIT_DIR, f'{nama}.csv')
    df.to_csv(path, index=False, encoding='utf-8-sig')
    tipe = 'TRAIN' if 'train' in nama else 'TEST '
    size = os.path.getsize(path)/1024/1024
    print(f'  ✅ [{tipe}] {nama:28s}: {len(df):6,} baris ({size:.1f} MB)')

print(f'\n✅ Semua tersimpan di: {SPLIT_DIR}')

## Cell 7 — Panduan Penggunaan di Notebook Eksperimen

In [ ]:
print('='*65)
print('PANDUAN LOAD DATA DI SETIAP NOTEBOOK EKSPERIMEN')
print('='*65)

panduan = [
    ('E-01 LR Manual',
     'manual_xgb_train.csv', 'manual_xgb_test.csv'),
    ('E-02 LR Gabungan',
     'gabungan_xgb_train.csv', 'gabungan_xgb_test.csv'),
    ('E-03 XGB NoTune Manual',
     'manual_xgb_train.csv', 'manual_xgb_test.csv'),
    ('E-04 XGB NoTune Gabungan',
     'gabungan_xgb_train.csv', 'gabungan_xgb_test.csv'),
    ('E-05 XGB Tuned Manual',
     'manual_xgb_train.csv', 'manual_xgb_test.csv'),
    ('E-06 XGB Tuned Gabungan',
     'gabungan_xgb_train.csv', 'gabungan_xgb_test.csv'),
    ('E-07 IndoBERT Manual',
     'manual_idb_train.csv', 'manual_idb_test.csv'),
    ('E-08 IndoBERT Gabungan',
     'gabungan_idb_train.csv', 'gabungan_idb_test.csv'),
]

print(f'\n{"Eksperimen":28s} {"Train":30s} {"Test"}')
print('-'*85)
for exp, train_f, test_f in panduan:
    print(f'{exp:28s} {train_f:30s} {test_f}')

print(f'''
Cara load di notebook eksperimen:

  SPLIT_DIR = os.path.join(BASE_DIR, 'splits')

  df_train = pd.read_csv(os.path.join(SPLIT_DIR, 'manual_xgb_train.csv'))
  df_test  = pd.read_csv(os.path.join(SPLIT_DIR, 'manual_xgb_test.csv'))

  X_train = df_train['text'].fillna('').values
  y_train = df_train['label_pks'].map(LABEL_MAP).values
  X_test  = df_test['text'].fillna('').values
  y_test  = df_test['label_pks'].map(LABEL_MAP).values
''')